# Surjectivity of the Exponential Map on $U(D)$


---

## Background

Time evolution in quantum mechanics is generated by Hermitian Hamiltonians, and the unitary operators that result are obtained by exponentiating them. If we strip out the factor of $i$ and keep the bare generator, it is skew-Hermitian, meaning it equals minus its own conjugate transpose. These generators form the Lie algebra written $\mathfrak{u}(D)$, and exponentiating any one of them lands inside the unitary group $U(D)$. A natural question is whether the reverse also holds, so that every unitary can be written as the exponential of some generator.

For the unitary group the answer is yes, and this reflects the fact that $U(D)$ is compact. Compactness is what makes the exponential map cover the whole group, and once it is dropped the property can fail. Over the real invertible matrices, for instance, exponentiating a real generator always produces a positive determinant, so no generator can ever reach a matrix whose determinant is negative.

The reason surjectivity holds is that a unitary is a normal operator, so the spectral theorem supplies an orthonormal basis of eigenvectors with eigenvalues of unit modulus, each of the form $e^{i\theta}$. Collecting the phases $\theta_j$ and weighting the corresponding eigenprojectors by them assembles a skew-Hermitian generator whose exponential reproduces the original unitary. Because each phase can be shifted by any multiple of $2\pi$ without changing its eigenvalue, many different generators map to the same unitary, so the map covers the group while being far from one to one.

The notebook diagonalizes Haar-random unitaries and checks that exponentiating the principal-logarithm generator returns each matrix.

## Mathematical background

Every unitary is a single exponential of a Hermitian generator. Because $U(D)$ is a compact connected Lie group, the exponential map

$$
\exp : \mathfrak{u}(D) \to U(D), \qquad X \mapsto e^X
$$

is surjective, so for any $U$ there is a Hermitian $H$ with $U = e^{-iH}$. The constructive reason is the spectral theorem. A unitary is normal, so it diagonalizes as $U = V\,\mathrm{diag}(e^{-i\theta_j})\,V^\dagger$ with real eigenphases $\theta_j$, and setting $H = V\,\mathrm{diag}(\theta_j)\,V^\dagger$ gives a Hermitian generator with $e^{-iH} = U$. The generator is not unique, since each $\theta_j$ is defined only modulo $2\pi$, but one always exists. This is what guarantees that any target unitary can be written as evolution under some Hamiltonian, the premise behind reading quantum gates as continuous-time dynamics.

## Motivation

The exponential map $\exp: \mathfrak{u}(D) \to U(D)$ connects the Lie algebra (infinitesimal generators) to the Lie group (finite unitaries). A crucial question: is every unitary $U$ reachable as $e^X$ for some generator $X$? For the unitary group the answer is **yes**, the exponential map is surjective. This is a special property of compact groups and fails for non-compact groups (e.g., $\mathrm{GL}(n, \mathbb{R})$ contains matrices with negative determinant, unreachable from $\exp$).

## Exercise

Show that every $U \in U(D)$ can be written as $U = \exp(X)$ for some $X \in \mathfrak{u}(D)$.

## Solution

### Step 1: Spectral decomposition of $U$

Since $U$ is unitary, it is **normal** ($UU^\dagger = U^\dagger U$), so the spectral theorem gives:

$$
U = \sum_{j=1}^D e^{i\theta_j} |v_j\rangle\langle v_j|,
$$

where $\{|v_j\rangle\}$ is an orthonormal eigenbasis and $\theta_j \in (-\pi, \pi]$ are the eigenphases.

### Step 2: Construct the generator

Define

$$
X = \sum_{j=1}^D i\theta_j\,|v_j\rangle\langle v_j|.
$$

### Step 3: Verify $X$ is skew-Hermitian

$$
X^\dagger = \sum_j (-i\theta_j)|v_j\rangle\langle v_j| = -X. \quad \checkmark
$$

So $X \in \mathfrak{u}(D)$.

### Step 4: Verify $\exp(X) = U$

The matrix exponential acts diagonally on the eigenbasis:

$$
\exp(X) = \sum_j \exp(i\theta_j)|v_j\rangle\langle v_j| = U. \quad \checkmark
$$

### Remark on uniqueness

The map is **not injective**: $X$ and $X + 2\pi i\, |v_j\rangle\langle v_j|$ produce the same $U$. The choice $\theta_j \in (-\pi, \pi]$ selects the principal logarithm, but other branches exist. This multi-valuedness reflects the compactness of $U(D)$.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp
from sympy import Matrix, I, exp, pi, simplify, eye

# Verify exp map surjectivity for SU(2):
# exp(-i*theta*n.sigma/2) covers all of SU(2)
theta = sp.Symbol('theta', real=True)

# Example: R_z(theta) = exp(-i*theta*sigma_z/2)
sz = Matrix([[1, 0], [0, -1]])
# Explicit formula: diag(e^{-itheta/2}, e^{itheta/2})
Rz = Matrix([[sp.exp(-I*theta/2), 0], [0, sp.exp(I*theta/2)]])
print(f'R_z(theta) = {Rz}')

# Verify det = 1 (element of SU(2))
det = simplify(Rz.det())
print(f'det(R_z) = {det}')
assert det == 1, 'Not in SU(2)!'
print('det = 1 verified: R_z in SU(2)')

# Verify unitarity: R_z * R_z^dag = I
prod = simplify(Rz * Rz.adjoint())
print(f'R_z * R_z^dag = {prod}')
assert prod == eye(2)
print('Unitarity verified!')

# Period: R_z(4*pi) = I (double cover of SO(3))
Rz_4pi = Rz.subs(theta, 4*pi)
print(f'\nR_z(4pi) = {simplify(Rz_4pi)}')
print('SU(2) is the double cover of SO(3): period = 4pi')

---
## Numerical Verification

## Takeaway

Every $U \in U(D)$ is the exponential of a skew-Hermitian generator, recovered by diagonalizing $U$ and taking the principal logarithm of each eigenphase, so $\exp$ maps $\mathfrak{u}(D)$ onto all of $U(D)$. The map is not injective, since shifting any eigenphase by $2\pi$ leaves $U$ unchanged.

In [ ]:
import numpy as np
from scipy.linalg import expm
from scipy.stats import unitary_group

np.random.seed(42)

for D in [2, 4, 8]:
    U = unitary_group.rvs(D)

    # Step 1: Diagonalize
    evals, evecs = np.linalg.eig(U)
    phases = np.angle(evals)  # θ_j ∈ (-π, π]

    # Step 2: Construct X = i Σ θ_j |v_j⟩⟨v_j|
    X = evecs @ np.diag(1j * phases) @ evecs.conj().T

    # Verify skew-Hermiticity
    assert np.allclose(X, -X.conj().T), "X is not skew-Hermitian!"

    # Verify exp(X) = U
    U_recovered = expm(X)
    err = np.linalg.norm(U_recovered - U)
    print(f"D={D}: ‖exp(X) − U‖ = {err:.2e}")
    assert err < 1e-10

print("\nSurjectivity verified: every Haar-random U = exp(X).")

## References

- Goodman and Wallach, *Symmetry, Representations, and Invariants*, Springer (2009).
- Milnor, *Curvatures of left invariant metrics on Lie groups*, Advances in Mathematics (1976).
- Płodzień, *Information Scrambling, Quantum Chaos, and Haar-Random States*, lecture notes (2025). [arXiv:2511.14397](https://arxiv.org/abs/2511.14397)